In [1]:
import pandas as pd
import os
import json
from datetime import datetime
import time
import shutil

In [2]:
# ============================================================================
# 1. Проверка и подготовка окружения
# ============================================================================
print("\n" + "="*70)
print(" ПОДГОТОВКА ОКРУЖЕНИЯ")
print("="*70)

# Путь к файлу
FILE_PATH = r"O:\extracted\mimodump-dataset.csv"

# Проверка существования файла
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Файл не найден: {FILE_PATH}")

# Проверка свободного места на диске
def check_disk_space():
    """Проверяет свободное место на диске и возвращает True, если достаточно места"""
    try:
        # Для Windows
        total, used, free = shutil.disk_usage("O:")
        free_gb = free / (1024**3)
        print(f"✅ Свободное место на диске O: {free_gb:.2f} ГБ")
        return free_gb > 10  # Нужно минимум 10 ГБ свободного места
    except:
        # Если не получается определить (например, на Linux)
        print(" Не удалось определить свободное место на диске")
        return True

if not check_disk_space():
    print(" НЕДОСТАТОЧНО МЕСТА НА ДИСКЕ! Завершение выполнения.")
    exit()

# Создаем директорию для результатов
OUTPUT_DIR = r"O:\extracted"
os.makedirs(OUTPUT_DIR, exist_ok=True)


 ПОДГОТОВКА ОКРУЖЕНИЯ
✅ Свободное место на диске O: 166.34 ГБ


In [3]:
# ============================================================================
# 2. Исследование структуры файла
# ============================================================================
print("\n" + "="*70)
print(" ИССЛЕДОВАНИЕ СТРУКТУРЫ ФАЙЛА")
print("="*70)

# Проверяем первые строки файла для определения структуры
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    first_lines = [f.readline().strip() for _ in range(20)]

print("\nПервые 10 строк файла:")
for i, line in enumerate(first_lines[:10], 1):
    print(f"{i:3d}: {line[:100]}{'...' if len(line) > 100 else ''}")

# Определяем разделитель и количество колонок
header_line = first_lines[0]
columns = [col.strip() for col in header_line.split(';')]
expected_cols = len(columns)

print(f"\nОбнаружено колонок в заголовке: {expected_cols}")
print(f"Колонки: {columns}")

# Проверка количества разделителей
print("\nПроверка количества ';' в первых 10 строках данных:")
for i, line in enumerate(first_lines[1:11], 2):
    count = line.count(';')
    status = "✓" if count == expected_cols - 1 else "✗"
    print(f"{i:3d}: {status} {count:2d} разделителей (ожидается {expected_cols - 1}) → {line[:60]}...")


 ИССЛЕДОВАНИЕ СТРУКТУРЫ ФАЙЛА

Первые 10 строк файла:
  1: name;price;current_price;lowest_price;msrp_price;image;item_url;category_name;category_url;available...
  2: tulup.si Slika na platnu Pokrajina desert sand;24;31.95;31.95;24;https://i.cdn.nrholding.net/5671293...
  3: ";tulup.si;tulupsi-slika-na-platnu-pokrajina-100078120788;2021-11-08 16:50:39.142181
  4: tulup.si Slika na platnu Sea coast landscape;32;34.95;34.95;32;https://i.cdn.nrholding.net/56648085;...
  5: ";tulup.si;tulupsi-slika-na-platnu-sea-coast-100073849383;2021-11-08 16:51:14.274117
  6: tulup.si Slika na platnu Slap landscape vode;35;34.95;34.95;35;https://i.cdn.nrholding.net/56635821;...
  7: ";tulup.si;tulupsi-slika-na-platnu-slap-landscape-100073848076;2021-11-08 16:51:25.548717
  8: tulup.si Slika na platnu Slap narava;24;22.95;22.95;24;https://i.cdn.nrholding.net/56695376;https://...
  9: ";tulup.si;tulupsi-slika-na-platnu-slap-narava-100073848132;2021-11-08 16:51:27.341345
 10: tulup.si Slika na platnu Sun

In [4]:
# ============================================================================
# 3. Загрузка тестовой выборки
# ============================================================================
print("\n" + "="*70)
print(" ЗАГРУЗКА ТЕСТОВОЙ ВЫБОРКИ")
print("="*70)

# Попытка загрузить небольшую выборку для проверки структуры
try:
    sample_df = pd.read_csv(
        FILE_PATH,
        sep=';',
        nrows=1000,
        on_bad_lines='skip',
        engine='python',
        encoding='utf-8',
        dtype=str,
        keep_default_na=True,
        na_values=['', 'NA', 'null', 'NULL']
    )
    
    print(f"ОК Успешно прочитано {len(sample_df):,} строк из 1000")
    print(f"Фактические колонки ({len(sample_df.columns)}):")
    for i, col in enumerate(sample_df.columns[:10], 1):
        print(f"  {i:2d}. {col}")
    if len(sample_df.columns) > 10:
        print(f"  ... и ещё {len(sample_df.columns) - 10} колонок")
    
    # Проверка наличия ключевых колонок
    key_cols = ['name', 'price', 'category_name', 'parsed_at']
    print("\nНаличие ключевых колонок:")
    for col in key_cols:
        status = "ОК" if col in sample_df.columns else "НЕОК"
        print(f"  {status} {col}")
    
except Exception as e:
    print(f"НЕОК Ошибка при чтении тестовой выборки: {type(e).__name__}: {e}")
    raise


 ЗАГРУЗКА ТЕСТОВОЙ ВЫБОРКИ
ОК Успешно прочитано 1,000 строк из 1000
Фактические колонки (21):
   1. name
   2. price
   3. current_price
   4. lowest_price
   5. msrp_price
   6. image
   7. item_url
   8. category_name
   9. category_url
  10. available
  ... и ещё 11 колонок

Наличие ключевых колонок:
  ОК name
  ОК price
  ОК category_name
  ОК parsed_at


In [ ]:
# ============================================================================
# 4. Обработка данных с восстановлением (ИСПРАВЛЕННЫЙ КОД)
# ============================================================================
print("\n" + "="*70)
print(" НАЧАЛО ОБРАБОТКИ ДАННЫХ С ВОССТАНОВЛЕНИЕМ")
print("="*70)

import pandas as pd
import os
import json
from datetime import datetime
import time
import gc

# Параметры
FILE_PATH = r"O:\extracted\mimodump-dataset.csv"
OUTPUT_DIR = r"O:\extracted"
CHUNK_SIZE = 100000
SAMPLE_FRACTION = 0.01
SAVE_INTERVAL = 5 * 60  # 5 минут

# Проверка наличия промежуточных файлов
sample_file = os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet")
profile_file = os.path.join(OUTPUT_DIR, "dataset_profile.json")

# Инициализация переменных
if os.path.exists(sample_file) and os.path.exists(profile_file):
    print(" Обнаружены промежуточные результаты. Восстанавливаем работу...")
    
    # Загружаем профиль
    with open(profile_file, "r", encoding="utf-8") as f:
        profile = json.load(f)
    
    # Восстанавливаем ТОЛЬКО счетчик обработанных строк
    total_rows = profile.get('total_rows_processed', 0)
    print(f"   Восстановлено: {total_rows:,} строк обработано")
    
    # Загружаем существующую выборку
    existing_sample = pd.read_parquet(sample_file)
    sample_rows = [existing_sample]
    print(f"   Загружено {len(existing_sample):,} строк в выборку")
    
    # Пропускаем уже обработанные строки
    skip_rows = total_rows
    print(f"   Пропускаем первые {skip_rows:,} строк...")
    
    # Инициализируем ПУСТЫЕ множества для новых данных (не пытаемся восстановить уникальные значения)
    unique_names = set()
    unique_categories = set()
    unique_brands = set()
    unique_dates = set()
    
else:
    print(" Нет промежуточных результатов. Начинаем с нуля...")
    total_rows = 0
    unique_names = set()
    unique_categories = set()
    unique_brands = set()
    unique_dates = set()
    sample_rows = []
    skip_rows = 0

last_save_time = time.time()

# Функция для безопасного сохранения (БЕЗ попытки сохранить множества)
def safe_save():
    """Безопасное сохранение промежуточных результатов"""
    try:
        # Сохраняем выборку
        if sample_rows:
            sample_df_temp = pd.concat(sample_rows, ignore_index=True)
            # Ограничиваем размер выборки в памяти (максимум 500k строк)
            if len(sample_df_temp) > 500000:
                sample_df_temp = sample_df_temp.sample(n=500000, random_state=42)
            sample_df_temp.to_parquet(
                os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet"), 
                compression="zstd", 
                index=False
            )
        
        # Сохраняем ТОЛЬКО счетчики и статус (не пытаемся сохранить множества)
        profile_temp = {
            "file_path": FILE_PATH,
            "total_rows_processed": total_rows,
            "sample_size": len(sample_df_temp) if sample_rows else 0,
            "saved_at": datetime.now().isoformat(),
            "status": "in_progress",
            "memory_usage_mb": sample_df_temp.memory_usage(deep=True).sum() / (1024**2) if sample_rows else 0
        }
        with open(os.path.join(OUTPUT_DIR, "dataset_profile.json"), "w", encoding="utf-8") as f:
            json.dump(profile_temp, f, ensure_ascii=False, indent=2)
        
        print(f"\n Промежуточные результаты сохранены ({datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")
        print(f"   Обработано строк: {total_rows:,}")
        print(f"   Размер выборки: {len(sample_df_temp):,} строк")
        return True
    except Exception as e:
        print(f" ОШИБКА СОХРАНЕНИЯ: {e}")
        return False

# Обработка файла по чанкам
try:
    start_time = datetime.now()
    chunks_processed = 0
    
    # Обработка с пропуском уже обработанных строк
    for chunk in pd.read_csv(
        FILE_PATH,
        sep=';',
        chunksize=CHUNK_SIZE,
        dtype='object',
        on_bad_lines='skip',
        engine='python',
        encoding='utf-8',
        skiprows=range(1, skip_rows + 1) if skip_rows > 0 else None
    ):
        chunks_processed += 1
        current_chunk_size = len(chunk)
        total_rows += current_chunk_size
        
        # Собираем уникальные значения ТОЛЬКО из новых данных
        if 'name' in chunk.columns:
            unique_names.update(chunk['name'].dropna().unique())
        if 'category_name' in chunk.columns:
            unique_categories.update(chunk['category_name'].dropna().unique())
        if 'brand_name' in chunk.columns:
            unique_brands.update(chunk['brand_name'].dropna().unique())
        if 'parsed_at' in chunk.columns:
            unique_dates.update(chunk['parsed_at'].dropna().unique())
        
        # Собираем случайную выборку
        sample_size = int(current_chunk_size * SAMPLE_FRACTION)
        if sample_size > 0 and len(chunk) > 0:
            sample_chunk = chunk.sample(n=min(sample_size, len(chunk)), random_state=42)
            sample_rows.append(sample_chunk)
        
        # Периодическое сохранение КАЖДЫЕ 10 ЧАНКОВ (а не по времени) для контроля памяти
        if chunks_processed % 10 == 0:
            safe_save()
            # Очищаем память
            gc.collect()
        
        # Прогресс-бар с индикацией памяти
        if chunks_processed % 5 == 0:
            mem_usage = sample_rows[-1].memory_usage(deep=True).sum() / (1024**2) if sample_rows else 0
            print(f"Обработано: {total_rows:,} строк | Память выборки: {mem_usage:.1f} MB...", end='\r')
        
        # Защита от переполнения памяти выборки
        if len(sample_rows) > 50:  # ~5M строк
            safe_save()
            sample_rows = [pd.concat(sample_rows, ignore_index=True).sample(n=100000, random_state=42)]
    
    # Финальное сохранение
    print("\n\n" + "="*70)
    print(" СОХРАНЕНИЕ ФИНАЛЬНЫХ РЕЗУЛЬТАТОВ")
    print("="*70)
    
    # Объединяем выборку и ограничиваем размер
    sample_df_final = pd.concat(sample_rows, ignore_index=True)
    if len(sample_df_final) > 200000:  # Максимум 200k строк для анализа
        sample_df_final = sample_df_final.sample(n=200000, random_state=42)
        print(f" Выборка уменьшена до 200,000 строк для экономии памяти")
    
    # Сохраняем выборку
    sample_parquet_path = os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet")
    sample_df_final.to_parquet(sample_parquet_path, compression="zstd", index=False)
    
    # Вычисляем уникальные значения ИЗ ВЫБОРКИ (а не из всего датасета)
    unique_names_count = sample_df_final['name'].nunique() if 'name' in sample_df_final.columns else 0
    unique_categories_count = sample_df_final['category_name'].nunique() if 'category_name' in sample_df_final.columns else 0
    unique_brands_count = sample_df_final['brand_name'].nunique() if 'brand_name' in sample_df_final.columns else 0
    unique_dates_count = sample_df_final['parsed_at'].nunique() if 'parsed_at' in sample_df_final.columns else 0
    
    # Сохраняем финальный профиль
    profile = {
        "file_path": FILE_PATH,
        "file_size_gb": round(os.path.getsize(FILE_PATH) / (1024**3), 2),
        "total_rows_processed": total_rows,
        "unique_names_in_sample": int(unique_names_count),
        "unique_categories_in_sample": int(unique_categories_count),
        "unique_brands_in_sample": int(unique_brands_count),
        "unique_dates_in_sample": int(unique_dates_count),
        "sample_size": len(sample_df_final),
        "sample_fraction": SAMPLE_FRACTION,
        "columns": list(sample_df_final.columns),
        "analysis_timestamp": datetime.now().isoformat(),
        "status": "completed",
        "processing_time": str(datetime.now() - start_time)
    }
    
    profile_path = os.path.join(OUTPUT_DIR, "dataset_profile.json")
    with open(profile_path, "w", encoding="utf-8") as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)
    
    print(f"\n Обработка завершена успешно!")
    print(f"   Обработано строк: {total_rows:,}")
    print(f"   Уникальных товаров в выборке: {unique_names_count:,}")
    print(f"   Уникальных категорий в выборке: {unique_categories_count}")
    print(f"   Размер финальной выборки: {len(sample_df_final):,} строк")
    print(f"   Время выполнения: {datetime.now() - start_time}")

except Exception as e:
    print(f"\n НЕОЖИДАННАЯ ОШИБКА: {type(e).__name__}: {e}")
    safe_save()
    print(" Промежуточные результаты сохранены. Запустите код снова для продолжения.")


 НАЧАЛО ОБРАБОТКИ ДАННЫХ С ВОССТАНОВЛЕНИЕМ
 Обнаружены промежуточные результаты. Восстанавливаем работу...
   Восстановлено: 84,900,000 строк обработано
   Загружено 120,000 строк в выборку
   Пропускаем первые 84,900,000 строк...
